In [15]:
import torch
import torch.nn as nn

class SSMCell(nn.Module):
    """
    Single SSM layer — CPU compatible.
    Equivalent in spirit to one Mamba block.

    State space model:
        h_t = A * h_{t-1} + B * x_t
        y_t = C * h_t + D * x_t

    A, B, C, D are learned parameters.
    h is the hidden state that carries memory across timesteps.
    """
    def __init__(self, d_input: int, d_state: int):
        super().__init__()
        self.d_state = d_state

        # Transition matrix — initialised close to identity
        # so gradients flow well early in training
        self.A = nn.Parameter(torch.eye(d_state) * 0.9)
        self.B = nn.Parameter(torch.randn(d_state, d_input) * 0.01)
        self.C = nn.Parameter(torch.randn(d_input, d_state) * 0.01)
        self.D = nn.Parameter(torch.ones(d_input))   # skip connection

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len, d_input)
        returns: (batch, seq_len, d_input)
        """
        B_batch, seq_len, d_input = x.shape
        h = torch.zeros(B_batch, self.d_state, device=x.device)
        outputs = []

        for t in range(seq_len):
            x_t = x[:, t, :]                          # (batch, d_input)
            h   = h @ self.A.T + x_t @ self.B.T       # (batch, d_state)
            y_t = h @ self.C.T + x_t * self.D         # (batch, d_input)
            outputs.append(y_t.unsqueeze(1))

        return torch.cat(outputs, dim=1)               # (batch, seq_len, d_input)


class LOBSequenceModel(nn.Module):
    """
    Full model for predicting next mid-price direction
    from a sequence of LOB features.

    Architecture:
        input features → linear projection → SSM → SSM → linear head → sigmoid
    """
    def __init__(self,
                 n_features: int = 1,    # just I, or more features
                 d_model:    int = 32,   # internal width
                 d_state:    int = 16,   # SSM hidden state size
                 n_layers:   int = 2):   # number of stacked SSM layers
        super().__init__()

        self.input_proj = nn.Linear(n_features, d_model)

        self.ssm_layers = nn.ModuleList([
            SSMCell(d_model, d_state) for _ in range(n_layers)
        ])

        self.norm_layers = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(n_layers)
        ])

        # Output: take last timestep, project to single probability
        self.output_head = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len, n_features)
        returns: (batch,) probabilities of upward move
        """
        out = self.input_proj(x)           # (batch, seq_len, d_model)

        for ssm, norm in zip(self.ssm_layers, self.norm_layers):
            residual = out
            out = ssm(out)
            out = norm(out + residual)     # residual connection + norm

        last = out[:, -1, :]               # take last timestep
        return self.output_head(last).squeeze(-1)  # (batch,)

In [16]:
# ── standard library ──────────────────────────────────
from dataclasses import dataclass, field
from typing import Literal, Optional
from collections import deque
import uuid

# ── numerical / data ──────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats

# ── machine learning ──────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ── data structures ───────────────────────────────────
from sortedcontainers import SortedDict

# ── visualisation ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from tqdm import tqdm

# ── verify ────────────────────────────────────────────
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch  : 2.11.0+cu128
CUDA     : True
Device   : cuda


In [17]:
from dataclasses import dataclass
from typing import Literal

OrderSide = Literal["buy", "sell"]

class Order:
  order_id: int
  side: OrderSide
  price: float
  quantity: float
  time:float
  order_id: str = field(default_factory= lambda: str(uuid.uuid4()))

  def make (cls, side: OrderSide, price: float, quantity: float, time: float):
    return cls(order_id=uuid.uuid4(), side=side, price=price, quantity=quantity, time=time)

In [18]:
from collections import deque
from sortedcontainers import SortedDict
from typing import Optional

class LimitOrderBook:
    def __init__(self):
        self.bids: SortedDict = SortedDict()
        self.asks: SortedDict = SortedDict()
        self.order_index: dict = {}

    # ── best quotes ──────────────────────────
    def best_bid(self) -> Optional[float]:
        return self.bids.keys()[-1] if self.bids else None

    def best_ask(self) -> Optional[float]:
        return self.asks.keys()[0] if self.asks else None

    # ── derived quantities ───────────────────
    def mid_price(self) -> Optional[float]:
        b, a = self.best_bid(), self.best_ask()
        if b is None or a is None:
            return None
        return (b + a) / 2.0

    def spread(self) -> Optional[float]:
        b, a = self.best_bid(), self.best_ask()
        if b is None or a is None:
            return None
        return a - b

    def queue_imbalance(self) -> Optional[float]:
        b, a = self.best_bid(), self.best_ask()
        if b is None or a is None:
            return None
        n_b = sum(o.quantity for o in self.bids[b])
        n_a = sum(o.quantity for o in self.asks[a])
        denom = n_b + n_a
        return 0.0 if denom == 0 else (n_b - n_a) / denom

    # ── helpers ──────────────────────────────
    def _book_for(self, side):
        return self.bids if side == "buy" else self.asks

    def _add_resting(self, order):
        book = self._book_for(order.side)
        if order.price not in book:
            book[order.price] = deque()
        book[order.price].append(order)
        self.order_index[order.order_id] = (order.side, order.price)

    def _clean_empty_level(self, book, price):
        if price in book and len(book[price]) == 0:
            del book[price]

    # ── matching engine ──────────────────────
    def _match_incoming(self, incoming):
        trades = []
        passive_book = self.asks if incoming.side == "buy" else self.bids

        while incoming.quantity > 0:
            if incoming.side == "buy":
                best = self.best_ask()
                if best is None or incoming.price < best:
                    break
            else:
                best = self.best_bid()
                if best is None or incoming.price > best:
                    break

            queue = passive_book.get(best, deque())
            if not queue:
                del passive_book[best]
                continue

            resting = queue[0]
            match_qty = min(incoming.quantity, resting.quantity)

            trade_price = resting.price
            trades.append((trade_price, match_qty,
                           incoming.order_id, resting.order_id))

            incoming.quantity -= match_qty
            resting.quantity -= match_qty

            if resting.quantity == 0:
                queue.popleft()
                self.order_index.pop(resting.order_id, None)
                if not queue:
                    passive_book.pop(best, None)

        # leftover becomes resting
        if incoming.quantity > 0:
            self._add_resting(incoming)

        return trades

    # ── cancellation ─────────────────────────
    def cancel_order(self, order_id: str) -> bool:
        info = self.order_index.get(order_id)
        if not info:
            return False

        side, price = info
        book = self.bids if side == 'buy' else self.asks
        dq = book.get(price, deque())

        newdq = deque([o for o in dq if o.order_id != order_id])

        if newdq:
            book[price] = newdq
        else:
            book.pop(price, None)

        self.order_index.pop(order_id, None)
        return True

In [19]:
def add_limit_order(self, order: Order) -> list[tuple]:
    if order.side == "buy":
        a = self.best_ask()
        if a is not None and order.price >= a:
            return self._match_incoming(order)
    else:
        b = self.best_bid()
        if b is not None and order.price <= b:
            return self._match_incoming(order)
    self._add_resting(order)
    return []

In [20]:
def _match_incoming(self, incoming: Order) -> list[tuple]:
    trades = []
    passive_book = self.asks if incoming.side == "buy" else self.bids

    while incoming.quantity > 0:
        if incoming.side == "buy":
            best = self.best_ask()
            if best is None or incoming.price < best:
                break
        else:
            best = self.best_bid()
            if best is None or incoming.price > best:
                break

        queue = passive_book.get(best, deque())
        if not queue:
            del passive_book[best]
            continue

        resting   = queue[0]
        match_qty = min(incoming.quantity, resting.quantity)

        trades.append((best, match_qty,
                       incoming.order_id, resting.order_id))

        incoming.quantity -= match_qty
        resting.quantity  -= match_qty

        if resting.quantity == 0:
            queue.popleft()
            self.order_index.pop(resting.order_id, None)
            self._clean_empty_level(passive_book, best)

    if incoming.quantity > 0:
        self._add_resting(incoming)

    return trades

In [21]:
from dataclasses import dataclass

@dataclass
class SimulationParams:
    # Poisson arrival rates (events per second)
    lambda_limit_buy:   float = 10.0
    lambda_limit_sell:  float = 10.0
    lambda_market_buy:  float = 3.0
    lambda_market_sell: float = 3.0
    lambda_cancel:      float = 5.0

    # Price grid
    tick_size:    float = 0.01
    n_levels:     int   = 5
    initial_mid:  float = 100.0

    # Order sizing
    mean_qty:  float = 100.0
    std_qty:   float = 20.0

    # How far from best quote new limit orders arrive (in ticks)
    depth_mean: float = 2.0
    depth_std:  float = 1.5


@dataclass
class MidPriceEvent:
    time:       float   # when the mid-price changed
    imbalance:  float   # I measured just before the change
    direction:  int     # +1 if price went up, -1 if down
    mid_before: float
    mid_after:  float
    n_bid:      float   # volume at best bid before change
    n_ask:      float   # volume at best ask before change

In [22]:
import numpy as np
from typing import Optional

class LOBSimulator:
    def __init__(self, params: SimulationParams, seed: int = 42):
        self.p    = params
        self.rng  = np.random.default_rng(seed)
        self.lob  = LimitOrderBook()
        self.time = 0.0

        # what we collect
        self.mid_price_events:   list[MidPriceEvent]    = []
        self.mid_history:        list[tuple[float,float]] = []
        self.imbalance_history:  list[tuple[float,float]] = []

        self._seed_book()

    # ── seed the book so it isn't empty at t=0 ───────
    def _seed_book(self):
        tick = self.p.tick_size
        mid  = self.p.initial_mid

        best_bid = round(np.floor(mid / tick) * tick, 10)
        best_ask = round(best_bid + tick, 10)

        for i in range(self.p.n_levels):
            bid_price = round(best_bid - i * tick, 10)
            bid_qty   = max(1.0, self.rng.normal(
                            self.p.mean_qty, self.p.std_qty))
            self.lob.add_limit_order(
                Order.make("buy", bid_price, bid_qty, 0.0))

            ask_price = round(best_ask + i * tick, 10)
            ask_qty   = max(1.0, self.rng.normal(
                            self.p.mean_qty, self.p.std_qty))
            self.lob.add_limit_order(
                Order.make("sell", ask_price, ask_qty, 0.0))

    # ── helpers ───────────────────────────────────────
    def _next_time(self, rate: float) -> float:
        return self.rng.exponential(1.0 / rate)

    def _sample_qty(self) -> float:
        return max(1.0, round(
            self.rng.normal(self.p.mean_qty, self.p.std_qty)))

    def _sample_limit_price(self, side: str) -> float:
        tick  = self.p.tick_size
        depth = max(0, round(abs(
            self.rng.normal(self.p.depth_mean, self.p.depth_std))))

        if side == "buy":
            best  = self.lob.best_bid() or (self.p.initial_mid - tick)
            price = best - depth * tick
        else:
            best  = self.lob.best_ask() or (self.p.initial_mid + tick)
            price = best + depth * tick

        return round(price, 10)

    def _sample_cancellable(self) -> Optional[str]:
        ids = list(self.lob.order_index.keys())
        if not ids:
            return None
        return ids[self.rng.integers(0, len(ids))]

    # ── core event loop ───────────────────────────────
    def run(self, n_events: int = 50_000):
        p = self.p

        for _ in range(n_events):

            # 1. how long until each event type fires?
            dt = {
                "limit_buy":   self._next_time(p.lambda_limit_buy),
                "limit_sell":  self._next_time(p.lambda_limit_sell),
                "market_buy":  self._next_time(p.lambda_market_buy),
                "market_sell": self._next_time(p.lambda_market_sell),
                "cancel":      self._next_time(p.lambda_cancel),
            }

            # 2. whichever fires soonest is what happens
            event = min(dt, key=dt.get)
            self.time += dt[event]

            # 3. snapshot BEFORE applying the event
            mid_before = self.lob.mid_price()
            I_before   = self.lob.queue_imbalance()
            bb         = self.lob.best_bid()
            ba         = self.lob.best_ask()
            n_b = (sum(o.quantity for o in self.lob.bids[bb])
                   if bb is not None else 0.0)
            n_a = (sum(o.quantity for o in self.lob.asks[ba])
                   if ba is not None else 0.0)

            # 4. apply the event
            if event == "limit_buy":
                self.lob.add_limit_order(Order.make(
                    "buy",
                    self._sample_limit_price("buy"),
                    self._sample_qty(), self.time))

            elif event == "limit_sell":
                self.lob.add_limit_order(Order.make(
                    "sell",
                    self._sample_limit_price("sell"),
                    self._sample_qty(), self.time))

            elif event == "market_buy":
                self.lob.add_market_order(
                    "buy", self._sample_qty(), self.time)

            elif event == "market_sell":
                self.lob.add_market_order(
                    "sell", self._sample_qty(), self.time)

            elif event == "cancel":
                oid = self._sample_cancellable()
                if oid:
                    self.lob.cancel_order(oid)

            # 5. did the mid-price change?
            mid_after = self.lob.mid_price()

            if (mid_before is not None
                    and mid_after is not None
                    and mid_after != mid_before
                    and I_before  is not None):

                self.mid_price_events.append(MidPriceEvent(
                    time       = self.time,
                    imbalance  = I_before,
                    direction  = 1 if mid_after > mid_before else -1,
                    mid_before = mid_before,
                    mid_after  = mid_after,
                    n_bid      = n_b,
                    n_ask      = n_a,
                ))

            # 6. record running history for plots
            if mid_after is not None:
                self.mid_history.append((self.time, mid_after))
            if I_before is not None:
                self.imbalance_history.append((self.time, I_before))

    # ── extract training data ─────────────────────────
    def get_training_data(self):
        """
        Returns X (imbalance) and y (direction) arrays.
        y=1 means price went up, y=0 means price went down.
        Matches the paper's equation (9).
        """
        X = np.array([e.imbalance for e in self.mid_price_events])
        y = np.array([1 if e.direction == 1 else 0
                      for e in self.mid_price_events])
        return X, y

In [24]:
# ── imports ───────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Literal, Optional
from collections import deque
from sortedcontainers import SortedDict
import uuid
import numpy as np

# ── types ─────────────────────────────────────────────
OrderSide = Literal["buy", "sell"]

# ─────────────────────────────────────────────────────
# Order
# ─────────────────────────────────────────────────────
@dataclass
class Order:
    side:     OrderSide
    price:    float
    quantity: float
    time:     float
    order_id: str = field(default_factory=lambda: str(uuid.uuid4()))

    @classmethod
    def make(cls, side: OrderSide, price: float,
             quantity: float, time: float) -> "Order":
        return cls(side=side, price=price,
                   quantity=quantity, time=time)

# ─────────────────────────────────────────────────────
# Limit Order Book
# ─────────────────────────────────────────────────────
class LimitOrderBook:
    def __init__(self):
        self.bids:        SortedDict = SortedDict()
        self.asks:        SortedDict = SortedDict()
        self.order_index: dict       = {}

    def best_bid(self) -> Optional[float]:
        return self.bids.keys()[-1] if self.bids else None

    def best_ask(self) -> Optional[float]:
        return self.asks.keys()[0] if self.asks else None

    def mid_price(self) -> Optional[float]:
        b, a = self.best_bid(), self.best_ask()
        if b is None or a is None:
            return None
        return (b + a) / 2.0

    def spread(self) -> Optional[float]:
        b, a = self.best_bid(), self.best_ask()
        if b is None or a is None:
            return None
        return a - b

    def queue_imbalance(self) -> Optional[float]:
        b, a = self.best_bid(), self.best_ask()
        if b is None or a is None:
            return None
        n_b   = sum(o.quantity for o in self.bids[b])
        n_a   = sum(o.quantity for o in self.asks[a])
        denom = n_b + n_a
        return 0.0 if denom == 0 else (n_b - n_a) / denom

    def _book_for(self, side: OrderSide) -> SortedDict:
        return self.bids if side == "buy" else self.asks

    def _add_resting(self, order: Order):
        book = self._book_for(order.side)
        if order.price not in book:
            book[order.price] = deque()
        book[order.price].append(order)
        self.order_index[order.order_id] = (order.side, order.price)

    def _clean_empty_level(self, book: SortedDict, price: float):
        if price in book and len(book[price]) == 0:
            del book[price]

    def _match_incoming(self, incoming: Order) -> list[tuple]:
        trades       = []
        passive_book = self.asks if incoming.side == "buy" else self.bids

        while incoming.quantity > 0:
            if incoming.side == "buy":
                best = self.best_ask()
                if best is None or incoming.price < best:
                    break
            else:
                best = self.best_bid()
                if best is None or incoming.price > best:
                    break

            queue = passive_book.get(best, deque())
            if not queue:
                del passive_book[best]
                continue

            resting   = queue[0]
            match_qty = min(incoming.quantity, resting.quantity)

            trades.append((best, match_qty,
                           incoming.order_id, resting.order_id))

            incoming.quantity -= match_qty
            resting.quantity  -= match_qty

            if resting.quantity == 0:
                queue.popleft()
                self.order_index.pop(resting.order_id, None)
                self._clean_empty_level(passive_book, best)

        if incoming.quantity > 0:
            self._add_resting(incoming)

        return trades

    def add_limit_order(self, order: Order) -> list[tuple]:
        if order.side == "buy":
            a = self.best_ask()
            if a is not None and order.price >= a:
                return self._match_incoming(order)
        else:
            b = self.best_bid()
            if b is not None and order.price <= b:
                return self._match_incoming(order)
        self._add_resting(order)
        return []

    def add_market_order(self, side: OrderSide,
                         quantity: float, time: float) -> list[tuple]:
        price = float("inf") if side == "buy" else float("-inf")
        order = Order.make(side=side, price=price,
                           quantity=quantity, time=time)
        return self._match_incoming(order)

    def cancel_order(self, order_id: str) -> bool:
        info = self.order_index.get(order_id)
        if not info:
            return False
        side, price = info
        book        = self._book_for(side)
        if price in book:
            book[price] = deque(
                o for o in book[price] if o.order_id != order_id)
            self._clean_empty_level(book, price)
        self.order_index.pop(order_id, None)
        return True

    def __repr__(self) -> str:
        if self.mid_price() is None:
            return "LOB | empty"
        return (f"LOB | best_bid={self.best_bid()} "
                f"best_ask={self.best_ask()} "
                f"mid={self.mid_price()} "
                f"I={self.queue_imbalance():.3f}")

# ─────────────────────────────────────────────────────
# Simulation dataclasses
# ─────────────────────────────────────────────────────
@dataclass
class SimulationParams:
    lambda_limit_buy:   float = 10.0
    lambda_limit_sell:  float = 10.0
    lambda_market_buy:  float = 3.0
    lambda_market_sell: float = 3.0
    lambda_cancel:      float = 5.0
    tick_size:          float = 0.01
    n_levels:           int   = 5
    initial_mid:        float = 100.0
    mean_qty:           float = 100.0
    std_qty:            float = 20.0
    depth_mean:         float = 2.0
    depth_std:          float = 1.5

@dataclass
class MidPriceEvent:
    time:       float
    imbalance:  float
    direction:  int
    mid_before: float
    mid_after:  float
    n_bid:      float
    n_ask:      float

# ─────────────────────────────────────────────────────
# Simulator
# ─────────────────────────────────────────────────────
class LOBSimulator:
    def __init__(self, params: SimulationParams, seed: int = 42):
        self.p    = params
        self.rng  = np.random.default_rng(seed)
        self.lob  = LimitOrderBook()
        self.time = 0.0

        self.mid_price_events:  list[MidPriceEvent]     = []
        self.mid_history:       list[tuple[float,float]] = []
        self.imbalance_history: list[tuple[float,float]] = []

        self._seed_book()

    def _seed_book(self):
        tick = self.p.tick_size
        mid  = self.p.initial_mid

        best_bid = round(np.floor(mid / tick) * tick, 10)
        best_ask = round(best_bid + tick, 10)

        for i in range(self.p.n_levels):
            bid_price = round(best_bid - i * tick, 10)
            bid_qty   = max(1.0, self.rng.normal(
                            self.p.mean_qty, self.p.std_qty))
            self.lob.add_limit_order(
                Order.make("buy", bid_price, bid_qty, 0.0))

            ask_price = round(best_ask + i * tick, 10)
            ask_qty   = max(1.0, self.rng.normal(
                            self.p.mean_qty, self.p.std_qty))
            self.lob.add_limit_order(
                Order.make("sell", ask_price, ask_qty, 0.0))

    def _next_time(self, rate: float) -> float:
        return self.rng.exponential(1.0 / rate)

    def _sample_qty(self) -> float:
        return max(1.0, round(
            self.rng.normal(self.p.mean_qty, self.p.std_qty)))

    def _sample_limit_price(self, side: str) -> float:
        tick  = self.p.tick_size
        depth = max(0, round(abs(
            self.rng.normal(self.p.depth_mean, self.p.depth_std))))

        if side == "buy":
            best  = self.lob.best_bid() or (self.p.initial_mid - tick)
            price = best - depth * tick
        else:
            best  = self.lob.best_ask() or (self.p.initial_mid + tick)
            price = best + depth * tick

        return round(price, 10)

    def _sample_cancellable(self) -> Optional[str]:
        ids = list(self.lob.order_index.keys())
        if not ids:
            return None
        return ids[self.rng.integers(0, len(ids))]

    def run(self, n_events: int = 50_000):
        p = self.p

        for _ in range(n_events):
            dt = {
                "limit_buy":   self._next_time(p.lambda_limit_buy),
                "limit_sell":  self._next_time(p.lambda_limit_sell),
                "market_buy":  self._next_time(p.lambda_market_buy),
                "market_sell": self._next_time(p.lambda_market_sell),
                "cancel":      self._next_time(p.lambda_cancel),
            }

            event      = min(dt, key=dt.get)
            self.time += dt[event]

            mid_before = self.lob.mid_price()
            I_before   = self.lob.queue_imbalance()
            bb         = self.lob.best_bid()
            ba         = self.lob.best_ask()
            n_b = (sum(o.quantity for o in self.lob.bids[bb])
                   if bb is not None else 0.0)
            n_a = (sum(o.quantity for o in self.lob.asks[ba])
                   if ba is not None else 0.0)

            if event == "limit_buy":
                self.lob.add_limit_order(Order.make(
                    "buy", self._sample_limit_price("buy"),
                    self._sample_qty(), self.time))

            elif event == "limit_sell":
                self.lob.add_limit_order(Order.make(
                    "sell", self._sample_limit_price("sell"),
                    self._sample_qty(), self.time))

            elif event == "market_buy":
                self.lob.add_market_order(
                    "buy", self._sample_qty(), self.time)

            elif event == "market_sell":
                self.lob.add_market_order(
                    "sell", self._sample_qty(), self.time)

            elif event == "cancel":
                oid = self._sample_cancellable()
                if oid:
                    self.lob.cancel_order(oid)

            mid_after = self.lob.mid_price()

            if (mid_before is not None
                    and mid_after  is not None
                    and mid_after  != mid_before
                    and I_before   is not None):

                self.mid_price_events.append(MidPriceEvent(
                    time       = self.time,
                    imbalance  = I_before,
                    direction  = 1 if mid_after > mid_before else -1,
                    mid_before = mid_before,
                    mid_after  = mid_after,
                    n_bid      = n_b,
                    n_ask      = n_a,
                ))

            if mid_after is not None:
                self.mid_history.append((self.time, mid_after))
            if I_before is not None:
                self.imbalance_history.append((self.time, I_before))

    def get_training_data(self):
        X = np.array([e.imbalance  for e in self.mid_price_events])
        y = np.array([1 if e.direction == 1 else 0
                      for e in self.mid_price_events])
        return X, y

# ─────────────────────────────────────────────────────
# Quick check
# ─────────────────────────────────────────────────────
print("Classes defined:", [c for c in
      ["Order","LimitOrderBook","SimulationParams",
       "MidPriceEvent","LOBSimulator"]
      if c in dir()])
print([m for m in dir(LimitOrderBook) if not m.startswith("__")])

Classes defined: ['Order', 'LimitOrderBook', 'SimulationParams', 'MidPriceEvent', 'LOBSimulator']
['_add_resting', '_book_for', '_clean_empty_level', '_match_incoming', 'add_limit_order', 'add_market_order', 'best_ask', 'best_bid', 'cancel_order', 'mid_price', 'queue_imbalance', 'spread']


In [27]:
params = SimulationParams(
    lambda_limit_buy   = 10.0,
    lambda_limit_sell  = 10.0,
    lambda_market_buy  = 3.0,
    lambda_market_sell = 3.0,
    lambda_cancel      = 5.0,
    tick_size          = 0.01,
    n_levels           = 5,
    initial_mid        = 100.0,
    mean_qty           = 100.0,
    std_qty            = 20.0,
)

sim = LOBSimulator(params, seed=42)
sim.run(n_events=50_000)

X, y = sim.get_training_data()

print(f"Total events run        : 50,000")
print(f"Mid-price changes found : {len(sim.mid_price_events)}")
print(f"Simulation time elapsed : {sim.time:.1f}s")
print(f"Imbalance range         : [{X.min():.3f}, {X.max():.3f}]")
print(f"Fraction upward moves   : {y.mean():.3f}")
print()
print(f"P(up | I >  0.3) = {y[X >  0.3].mean():.3f}  ← should be > 0.5")
print(f"P(up | I < -0.3) = {y[X < -0.3].mean():.3f}  ← should be < 0.5")

Total events run        : 50,000
Mid-price changes found : 25
Simulation time elapsed : 1618.9s
Imbalance range         : [-0.996, 1.000]
Fraction upward moves   : 0.560

P(up | I >  0.3) = 1.000  ← should be > 0.5
P(up | I < -0.3) = 0.000  ← should be < 0.5
